**Data Cleaning and Preprocessing**

In [1]:
import pandas as pd
import numpy as np
import re

def clean_support_tickets(file_path):
    print(f"--- Starting Data Cleaning for: {file_path} ---")

    # 1. Load Dataset
    df = pd.read_csv(file_path)
    initial_shape = df.shape

    # 2. Remove Duplicate Records
    # Ensuring we evaluate uniqueness based on structural metadata and content
    unique_cols = ['Ticket_Subject', 'Ticket_Description', 'Customer_Email', 'Submission_Date']
    df.drop_duplicates(subset=unique_cols, keep='first', inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Removed {initial_shape[0] - df.shape[0]} duplicate row(s).")

    # 3. Handle Missing Values (Imputation Strategy)
    # Critical text features cannot be null for NLP. Metadata uses structural defaults.
    df['Ticket_Subject'] = df['Ticket_Subject'].fillna('No Subject Provided')
    df['Ticket_Description'] = df['Ticket_Description'].fillna('No Description Provided')
    df['Issue_Category'] = df['Issue_Category'].fillna('General Inquiry')
    df['Ticket_Channel'] = df['Ticket_Channel'].fillna('Web Form')

    # Numerical Imputation (Use median of the category to avoid skewing resolution proxies)
    if df['Resolution_Time_Hours'].isnull().any():
        df['Resolution_Time_Hours'] = df.groupby('Issue_Category')['Resolution_Time_Hours'].transform(
            lambda x: x.fillna(x.median())
        )

    # 4. Standardize Dates & Temporal Values
    # Ensuring absolute datetime parsing so temporal calculations don't crash
    df['Submission_Date'] = pd.to_datetime(df['Submission_Date'], errors='coerce')
    # Default broken dates to a placeholder fallback or mode if parsing fails
    df['Submission_Date'] = df['Submission_Date'].fillna(df['Submission_Date'].mode()[0])

    # 5. Clean and Normalize Text Fields (Preserving semantic context for Stage 2)
    def sanitize_text(text):
        if not isinstance(text, str):
            return ""
        # Convert to lowercase
        text = text.lower()
        # Remove random synthetic noise (e.g., trailing isolated Faker filler words/gibberish)
        # Replacing multiple spaces or newlines with a single space
        text = re.sub(r'\s+', ' ', text)
        # Trim leading and trailing whitespace
        return text.strip()

    df['cleaned_subject'] = df['Ticket_Subject'].apply(sanitize_text)
    df['cleaned_description'] = df['Ticket_Description'].apply(sanitize_text)

    # Combine subject and description into a unified semantic body text for NLP models
    df['combined_text'] = df['cleaned_subject'] + " | " + df['cleaned_description']

    # 6. Handle Outliers in Numerical Metrics (Winsorization)
    # Extremely large resolution times might skew your Stage 1 Z-scores heavily.
    # Cap upper outliers at the 99th percentile to retain data structure safely.
    upper_limit = df['Resolution_Time_Hours'].quantile(0.99)
    df['Resolution_Time_Hours'] = np.where(
        df['Resolution_Time_Hours'] > upper_limit,
        upper_limit,
        df['Resolution_Time_Hours']
    )

    # 7. Standardize Categorical Labels
    # Enforce exact string casing to prevent 'critical' and 'Critical' from bifurcating categories
    df['Priority_Level'] = df['Priority_Level'].astype(str).str.capitalize().str.strip()
    df['Issue_Category'] = df['Issue_Category'].astype(str).str.title().str.strip()

    print(f"Data Cleaning Completed. Final Data Shape: {df.shape}\n")
    return df

# --- Integration Example ---
# Clean your primary dataset
cleaned_df = clean_support_tickets("enhanced_customer_support_data.csv")

# Now map human labels safely using your sanitized data
priority_map = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
cleaned_df["assigned_priority_num"] = cleaned_df["Priority_Level"].map(priority_map)

--- Starting Data Cleaning for: enhanced_customer_support_data.csv ---
Removed 0 duplicate row(s).
Data Cleaning Completed. Final Data Shape: (20000, 15)



In [2]:
cleaned_df.columns

Index(['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject',
       'Ticket_Description', 'Issue_Category', 'Priority_Level',
       'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours',
       'Assigned_Agent', 'Satisfaction_Score', 'cleaned_subject',
       'cleaned_description', 'combined_text', 'assigned_priority_num'],
      dtype='object')

In [3]:
# 3. Engineer corporate domain tier weighting (as specified in your Kaggle data summary)
def extract_domain_tier(email):
    if pd.isna(email):
        return 0
    # Flag high-value corporate accounts ending in .org or corporate keywords
    if "@enterprise.org" in email or "@company.com" in email or "@tech.io" in email:
        return 4  # Premium / Enterprise tier weight
    else:
        return 1  # Standard consumer tier weight

cleaned_df["customer_tier"] = cleaned_df["Customer_Email"].apply(extract_domain_tier)
print(cleaned_df.shape)
print(cleaned_df.columns)

(20000, 17)
Index(['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject',
       'Ticket_Description', 'Issue_Category', 'Priority_Level',
       'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours',
       'Assigned_Agent', 'Satisfaction_Score', 'cleaned_subject',
       'cleaned_description', 'combined_text', 'assigned_priority_num',
       'customer_tier'],
      dtype='object')


**Signal 1: Rule based NLP Scoring**

In [4]:
import re
import pandas as pd
import numpy as np

def generate_nlp_urgency_signal(df):
    """
    Stage 1 - Signal 1: Rule-Based NLP Urgency Scoring
    Fuses keyword density, escalation indicators, and structural weights
    to calculate an objective intensity score independent of assigned labels.
    """
    print("--- Generating Signal 1: Rule-Based NLP Urgency ---")

    # 1. Define Lexicon Matrices
    critical_keywords = [
        r'\bbreak(ing|s)?\b', r'\bcrash(ed|es|ing)?\b', r'\bdown\b', r'\bbroken\b',
        r'\bbreach\b', r'\bexploit\b', r'\bvulnerability\b', r'\bleak\b',
        r'\bdisaster\b', r'\bemergency\b', r'\bparalyzed\b', r'\bcorrupted\b',
        r'\bstop(ped|ping)? work\b', r'\bproduction halt\b', r'\bdata loss\b',
        r'\bunusable\b', r'\bblocker\b', r'\bdenial of service\b', r'\bdos\b',
        r"\bcan't login\b", r"\bcannot login\b", r"\bunable to login\b",
        r"\bunable to access\b", r"\bcannot access\b", r"\blocked out\b", r"\bpayment failed\b",
        r"\baccount suspended\b", r"\bcan't use\b", r"\bcannot use\b", r"\bunable to use\b",
        r"\bnot receiving\b", r"\bstuck\b", r"\boutage\b", r"\bservice unavailable\b"
    ]

    # Actionable escalations or commercial pressure
    escalation_phrases = [
        r'\bescalat(e|ed|ing|ion)\b', r'\blegal\b', r'\bsue\b', r'\blawyer\b',
        r'\brefund\b', r'\bcancel\b', r'\bchurn\b', r'\blose money\b',
        r'\bbreach of contract\b', r'\bsla\b', r'\bmanager\b', r'\bvp\b',
        r'\bceo\b', r'\bdirector\b', r'\burgent\b', r'\basap\b', r'\bimmediately\b'
    ]

    # Low-priority, informative, or casual interactions
    low_priority_keywords = [
        r'\bcurious\b', r'\bfeedback\b', r'\bsuggestion\b', r'\bfeature request\b', r'\bupdate profile\b',
        r'\btutorial\b', r'\bdocumentation\b', r'\binquiry\b', r'\bjust wondering\b'
    ]

    crit_regex = re.compile('|'.join(critical_keywords), flags=re.IGNORECASE)
    esc_regex = re.compile('|'.join(escalation_phrases), flags=re.IGNORECASE)
    low_regex = re.compile('|'.join(low_priority_keywords), flags=re.IGNORECASE)

    # 2. Extract Count/Density Signals
    texts = df['combined_text'].fillna("").astype(str)

    df['nlp_crit_count'] = texts.apply(lambda x: len(crit_regex.findall(x)))
    df['nlp_esc_count'] = texts.apply(lambda x: len(esc_regex.findall(x)))
    df['nlp_low_count'] = texts.apply(lambda x: len(low_regex.findall(x)))

    # 3. Negation Tracking
    negated_urgency_patterns = [r'\bnot\s+urgent\b',r'\bnot\s+critical\b',r'\bnot\s+severe\b',
    r'\bnot\s+blocked\b',r'\bnot\s+broken\b',r'\bnot\s+down\b',r'\bnot\s+crashing\b',
    r'\bno\s+issue\b',r'\bno\s+issues\b',r'\bno\s+problem\b',r'\bno\s+problems\b',
    r'\bworking\s+fine\b',r'\bworks\s+fine\b',r'\bresolved\b',r'\bfixed\b'
    ]
    negation_regex = re.compile('|'.join(negated_urgency_patterns), flags=re.IGNORECASE)
    df['nlp_negation_count'] = texts.apply(lambda x: len(negation_regex.findall(x)))

    # 4. Calculate Raw Urgency Score
    raw_score = (
        (df['nlp_crit_count'] * 3.0) +
        (df['nlp_esc_count'] * 2.0) -
        (df['nlp_low_count'] * 1.5)
    )
    raw_score = raw_score - (df['nlp_negation_count'] * 0.5)
    df['nlp_raw_score'] = raw_score + (df['customer_tier'] * 0.75)

    # 5. Continuous Normalization to 1-4 scale
    min_s = df['nlp_raw_score'].min()
    max_s = df['nlp_raw_score'].max()

    if max_s == min_s:
        df['nlp_normalized_score'] = 1.0
    else:
        df['nlp_normalized_score'] = 1.0 + 3.0 * ((df['nlp_raw_score'] - min_s) / (max_s - min_s))

    # 6. REALISTIC FIX: Threshold-Based Binning (No more artificial balancing!)
    # We define logical cutoffs on our 1.0 to 4.0 normalized score.
    # This is a fix done after using quartile cut, which balanced all four classes forcefully!
    score = df['nlp_normalized_score']

    conditions = [
        (score < 1.8),                              # Low-risk / standard inquiries
        (score >= 1.8) & (score < 2.5),             # Medium priority operational issues
        (score >= 2.5) & (score < 3.4),             # High priority / Escalation potential
        (score >= 3.4)                              # Critical / System down / Premium Tier emergencies
    ]

    choices = [1, 2, 3, 4]

    # np.select applies these conditions sequentially and handles duplicate values flawlessly
    df['nlp_inferred_severity'] = np.select(conditions, choices, default=1).astype(int)

    # Clean up tracking columns
    drop_cols = ['nlp_crit_count', 'nlp_esc_count', 'nlp_low_count', 'nlp_negation_count']
    df.drop(columns=drop_cols, inplace=True)

    print("Signal 1 successfully engineered with realistic threshold distributions.")
    print(df['nlp_inferred_severity'].value_counts().sort_index().to_string())
    print("-" * 40 + "\n")

    return df

# --- Execution ---
cleaned_df = generate_nlp_urgency_signal(cleaned_df)

--- Generating Signal 1: Rule-Based NLP Urgency ---
Signal 1 successfully engineered with realistic threshold distributions.
nlp_inferred_severity
1    10071
2     8388
3     1458
4       83
----------------------------------------



In [5]:
cleaned_df.columns

Index(['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject',
       'Ticket_Description', 'Issue_Category', 'Priority_Level',
       'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours',
       'Assigned_Agent', 'Satisfaction_Score', 'cleaned_subject',
       'cleaned_description', 'combined_text', 'assigned_priority_num',
       'customer_tier', 'nlp_raw_score', 'nlp_normalized_score',
       'nlp_inferred_severity'],
      dtype='object')

In [6]:
cleaned_df.drop(columns=['nlp_raw_score', 'nlp_normalized_score'], inplace=True)
cleaned_df.columns

Index(['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject',
       'Ticket_Description', 'Issue_Category', 'Priority_Level',
       'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours',
       'Assigned_Agent', 'Satisfaction_Score', 'cleaned_subject',
       'cleaned_description', 'combined_text', 'assigned_priority_num',
       'customer_tier', 'nlp_inferred_severity'],
      dtype='object')

**Signal 2: Resolution Time**

In [7]:

def generate_resolution_time_signal(df):
    """
    Stage 1 - Signal 2: Resolution-Time Regression / Outlier Proxy
    Calculates category-normalized Z-scores for resolution hours to
    infer true severity based on operational complexity.
    """
    print("--- Generating Signal 2: Resolution-Time Proxy ---")

    # 1. Calculate Category-Specific Means and Standard Deviations
    # This prevents global variations from distorting localized category baselines

    # Log transform to reduce influence of extreme ticket durations
    df['log_resolution_time'] = np.log1p(df['Resolution_Time_Hours'])

    grouped = df.groupby('Issue_Category')['log_resolution_time']

    df['category_mean_hours'] = grouped.transform('mean')
    df['category_std_hours'] = grouped.transform('std')

    df['res_time_zscore'] = (
        (df['log_resolution_time'] - df['category_mean_hours']) /
        (df['category_std_hours'] + 1e-5)
    )

    # 3. Apply Realistic Threshold-Based Binning via np.select
    # Standard deviations guide our operational severity tiers:
    # Z < 0: Faster than average (Low operational friction)
    # 0 <= Z < 1: Standard handling window (Medium friction)
    # 1 <= Z < 2.5: Significant delay / Escalation path (High friction)
    # Z >= 2.5: Severe outlier / Blocked or complex resolution (Critical friction)
    z = df['res_time_zscore']

    conditions = [
        (z < 0.0),
        (z >= 0.0) & (z < 1.2),
        (z >= 1.2) & (z < 2.6),
        (z >= 2.6)
    ]

    choices = [1, 2, 3, 4]

    df['res_time_inferred_severity'] = np.select(conditions, choices, default=1).astype(int)

    # Clean up intermediate baseline columns to keep the dataframe pristine
    drop_cols = ['category_mean_hours', 'category_std_hours']
    df.drop(columns=drop_cols, inplace=True)

    print("Signal 2 successfully engineered.")
    print(df['res_time_inferred_severity'].value_counts().sort_index().to_string())
    print("-" * 40 + "\n")

    return df

# --- Execution ---
cleaned_df = generate_resolution_time_signal(cleaned_df)

--- Generating Signal 2: Resolution-Time Proxy ---
Signal 2 successfully engineered.
res_time_inferred_severity
1    9171
2    8386
3    2443
----------------------------------------



In [8]:
cleaned_df.columns

Index(['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject',
       'Ticket_Description', 'Issue_Category', 'Priority_Level',
       'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours',
       'Assigned_Agent', 'Satisfaction_Score', 'cleaned_subject',
       'cleaned_description', 'combined_text', 'assigned_priority_num',
       'customer_tier', 'nlp_inferred_severity', 'log_resolution_time',
       'res_time_zscore', 'res_time_inferred_severity'],
      dtype='object')

In [9]:
cleaned_df.drop(columns=['res_time_zscore'], inplace=True)
cleaned_df.columns

Index(['Ticket_ID', 'Customer_Name', 'Customer_Email', 'Ticket_Subject',
       'Ticket_Description', 'Issue_Category', 'Priority_Level',
       'Ticket_Channel', 'Submission_Date', 'Resolution_Time_Hours',
       'Assigned_Agent', 'Satisfaction_Score', 'cleaned_subject',
       'cleaned_description', 'combined_text', 'assigned_priority_num',
       'customer_tier', 'nlp_inferred_severity', 'log_resolution_time',
       'res_time_inferred_severity'],
      dtype='object')

**Signal 3: Semantics Embedding Clustering**

In [10]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def generate_embedding_anchor_signal(df, model_name='all-MiniLM-L6-v2'):
    """
    Stage 1 - Signal 3: Anchor-Based Embedding Clustering

    Uses multiple semantic anchor examples per severity level and
    assigns severity based on average cosine similarity to each
    severity group.
    """

    print(f"--- Generating Signal 3: Anchor-Based Embeddings ({model_name}) ---")

    # ==========================================================
    # 1. Initialize Sentence Transformer
    # ==========================================================
    model = SentenceTransformer(model_name)

    # ==========================================================
    # 2. Multi-Anchor Severity Seeds
    # ==========================================================
    anchors = {
        1: [
            "General question about account settings and configuration.",
            "Documentation clarification or tutorial assistance request.",
            "User inquiry seeking information about product usage.",
            "Casual feedback or non-urgent customer question.",
            "Request for guidance regarding standard functionality."
        ],

        2: [
            "Minor interface bug affecting a single user.",
            "Routine operational issue requiring standard troubleshooting.",
            "Feature request or enhancement suggestion.",
            "Small configuration problem with limited impact.",
            "Non-critical defect with available workaround."
        ],

        3: [
            "Escalated issue requiring management attention.",
            "Repeated system errors impacting productivity.",
            "Business workflow disruption affecting multiple users.",
            "Service degradation causing operational inefficiency.",
            "Significant issue requiring urgent investigation."
        ],

        4: [
            "Production system outage affecting all customers.",
            "Critical security breach exposing sensitive data.",
            "Database corruption causing severe data loss.",
            "Payment or core business service completely unavailable.",
            "Blocker preventing normal business operations."
        ]
    }

    # ==========================================================
    # 3. Flatten Anchor Structure
    # ==========================================================
    anchor_texts = []
    anchor_severities = []

    for severity, examples in anchors.items():
        for text in examples:
            anchor_texts.append(text)
            anchor_severities.append(severity)

    print("Encoding semantic anchor seeds...")

    anchor_embeddings = model.encode(
        anchor_texts,
        show_progress_bar=False
    )

    # ==========================================================
    # 4. Encode Ticket Texts
    # ==========================================================
    ticket_texts = (
        df['combined_text']
        .fillna("no context available")
        .astype(str)
        .tolist()
    )

    print(
        f"Encoding {len(ticket_texts)} customer support tickets..."
    )

    ticket_embeddings = model.encode(
        ticket_texts,
        batch_size=32,
        show_progress_bar=True
    )

    # ==========================================================
    # 5. Similarity Matrix
    # ==========================================================
    print("Calculating semantic proximity matrix via Cosine Similarity...")

    similarity_matrix = cosine_similarity(
        ticket_embeddings,
        anchor_embeddings
    )

    # ==========================================================
    # 6. Average Similarity per Severity Group
    # ==========================================================
    severity_scores = []

    for severity in [1, 2, 3, 4]:

        indices = [
            i for i, s in enumerate(anchor_severities)
            if s == severity
        ]

        avg_similarity = similarity_matrix[:, indices].mean(axis=1)

        severity_scores.append(avg_similarity)

    severity_scores = np.column_stack(severity_scores)

    # ==========================================================
    # 7. Assign Severity via Argmax
    # ==========================================================
    best_indices = np.argmax(severity_scores, axis=1)

    df['emb_inferred_severity'] = best_indices + 1

    # ==========================================================
    # 8. Confidence Metric
    # ==========================================================
    df['emb_anchor_confidence'] = np.max(
        severity_scores,
        axis=1
    )

    print("Signal 3 successfully engineered via semantic anchor mapping.")
    print(df['emb_inferred_severity'].value_counts().sort_index().to_string())
    print("-" * 40 + "\n")

    return df


# --- Execution ---
cleaned_df = generate_embedding_anchor_signal(cleaned_df)

--- Generating Signal 3: Anchor-Based Embeddings (all-MiniLM-L6-v2) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding semantic anchor seeds...
Encoding 20000 customer support tickets...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Calculating semantic proximity matrix via Cosine Similarity...
Signal 3 successfully engineered via semantic anchor mapping.
emb_inferred_severity
1    4531
2    4795
3    2459
4    8215
----------------------------------------



In [11]:
# Cross-tabulate Signal 1 and Signal 3 to see how they align
cross_tab = pd.crosstab(
    cleaned_df['nlp_inferred_severity'],
    cleaned_df['emb_inferred_severity'],
    margins=True
)
print("--- Cross-Tabulation: Rule-Based vs. Embedding Anchors ---")
print(cross_tab)

--- Cross-Tabulation: Rule-Based vs. Embedding Anchors ---
emb_inferred_severity     1     2     3     4    All
nlp_inferred_severity                               
1                      3035  2275  1345  3416  10071
2                      1405  1890  1066  4027   8388
3                        90   564    48   756   1458
4                         1    66     0    16     83
All                    4531  4795  2459  8215  20000


In [12]:
# Cross-tabulate Signal 2 and Signal 3 to see how they align
cross_tab = pd.crosstab(
    cleaned_df['res_time_inferred_severity'],
    cleaned_df['emb_inferred_severity'],
    margins=True
)
print("--- Cross-Tabulation: Rule-Based vs. Embedding Anchors ---")
print(cross_tab)

--- Cross-Tabulation: Rule-Based vs. Embedding Anchors ---
emb_inferred_severity          1     2     3     4    All
res_time_inferred_severity                               
1                           2035  2199  1168  3769   9171
2                           1926  1984  1003  3473   8386
3                            570   612   288   973   2443
All                         4531  4795  2459  8215  20000


In [13]:
cleaned_df.to_csv("cleaned_df_3_signals.csv", index=False)
print("CSV saved successfully!")

CSV saved successfully!


**Signal 4: LLM Zero shot Scoring**

In [14]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, Phi3ForCausalLM
from tqdm import tqdm

def generate_decoder_llm_signal(
    df,
    model_id="microsoft/Phi-3-mini-4k-instruct",
    batch_size=8
):
    """
    Stage 1 - Signal 4: Decoder-Only Zero-Shot Severity Scoring (Native Library Fix)
    Leverages a local generative LLM (Phi-3) using native transformers classes
    to bypass out-of-date remote code bugs entirely.
    """
    print(f"--- Generating Signal 4: Generative LLM Scoring ({model_id}) ---")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading weights to device: {device.upper()}")

    # 1. Initialize Tokenizer natively
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=False
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 2. Initialize Model natively using the explicit Phi3 class
    print("Loading native Phi3 architecture layers...")
    model = Phi3ForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=False
    )

    # 3. Define the Clean Structural Prompt
    instruction_prompt = (
        "You are an expert customer operations auditor. Analyze the following support ticket text "
        "and determine its objective urgency severity based strictly on this scale:\n"
        "Severity 1:\n"
        "- Information request\n"
        "- Documentation question\n"
        "- How-to guidance\n"
        "- General inquiry\n"
        "- Feedback or suggestion\n"
        "- No operational impact\n\n"

        "Severity 2:\n"
        "- Minor bug\n"
        "- Isolated user issue\n"
        "- Routine support request\n"
        "- Small functionality problem\n"
        "- Workaround exists\n"
        "- Limited business impact\n\n"

        "Severity 3:\n"
        "- Multiple users affected\n"
        "- Significant workflow disruption\n"
        "- Escalation requested\n"
        "- Important functionality unavailable\n"
        "- Business productivity impacted\n"
        "- No complete outage\n\n"

        "Severity 4:\n"
        "- Production outage\n"
        "- Critical security incident\n"
        "- Data loss or corruption\n"
        "- Core business process blocked\n"
        "- Service unavailable for many users\n"
        "- Immediate business risk\n\n"

       "Respond with EXACTLY one character: 1, 2, 3, or 4. DO NOT USE ANY TEXT."
    )

    # Pre-tokenize target tokens to track cross-entropy logit spaces
    target_tokens = ["1", "2", "3", "4"]
    target_ids = [
        tokenizer.encode(t, add_special_tokens=False)[-1]
        for t in target_tokens
    ]

    ticket_texts = (
        df['combined_text']
        .fillna("no context available")
        .astype(str)
        .tolist()
    )

    inferred_severities = []

    print("Processing tickets via causal text generation matrix...")

    for start_idx in tqdm(range(0, len(ticket_texts), batch_size)):

        batch_texts = ticket_texts[start_idx:start_idx + batch_size]

        prompts = []

        for text in batch_texts:
            messages = [
                {
                    "role": "user",
                    "content": (
                        f"{instruction_prompt}\n\n"
                        f"Ticket Text: {text}\n"
                        f"Severity Score:"
                    )
                }
            ]

            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            prompts.append(prompt)

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():

            outputs = model(**inputs)

            # Shape: [batch_size, vocab_size]
            next_token_logits = outputs.logits[:, -1, :]

            # Shape: [batch_size, 4]
            target_logits = next_token_logits[:, target_ids]

            probabilities = torch.softmax(
                target_logits,
                dim=-1
            ).cpu().numpy()

            batch_predictions = np.argmax(
                probabilities,
                axis=1
            ) + 1

            inferred_severities.extend(
                batch_predictions.tolist()
            )

    # 4. Integrate back to dataframe
    df['llm_inferred_severity'] = inferred_severities

    print("Signal 4 successfully generated via Decoder Reasoning.")
    print(
        df['llm_inferred_severity']
        .value_counts()
        .sort_index()
        .to_string()
    )
    print("-" * 40 + "\n")

    return df


# --- Execution ---
cleaned_df = generate_decoder_llm_signal(cleaned_df)

--- Generating Signal 4: Generative LLM Scoring (microsoft/Phi-3-mini-4k-instruct) ---
Loading weights to device: CUDA


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Processing tickets via causal text generation matrix...


100%|██████████| 2500/2500 [52:28<00:00,  1.26s/it]

Signal 4 successfully generated via Decoder Reasoning.
llm_inferred_severity
1    2303
2    5932
3    9662
4    2103
----------------------------------------



In [15]:
# Save dataframe to CSV
cleaned_df.to_csv("cleaned_df_with_llm_severity.csv", index=False)

print("CSV saved successfully!")

CSV saved successfully!


**Consensus Signal Synthesis**

In [16]:
import numpy as np
import pandas as pd

def synthesize_consensus_labels(file_path):
    print(f"--- Starting Stage 1 Label Synthesis for: {file_path} ---")

    # 1. Load your pre-computed 4-signal dataframe
    df = pd.read_csv(file_path)

    # 2. Define the optimized weight matrix
    # Language models get 70% of the total vote weight; structural proxies get 30%
    w_nlp = 0.05
    w_res = 0.05
    w_emb = 0.25
    w_llm = 0.65

    # Verification sanity check to ensure our configuration sums to exactly 1.0
    assert np.isclose(w_nlp + w_res + w_emb + w_llm, 1.0), "Weights must sum to exactly 1.0!"

    # 3. Calculate the continuous weighted score
    df['consensus_continuous_score'] = (
        (df['nlp_inferred_severity'] * w_nlp) +
        (df['res_time_inferred_severity'] * w_res) +
        (df['emb_inferred_severity'] * w_emb) +
        (df['llm_inferred_severity'] * w_llm)
    )
    print(df['consensus_continuous_score'].isna().sum())

    print(
        np.isinf(
            df['consensus_continuous_score']
        ).sum()
    )
    # 4. Round to the nearest integer to derive the discrete pseudo-label (y*)
    df['consensus_severity_label'] = np.round(df['consensus_continuous_score']).astype(int)

    # Clamp values just in case any floating-point anomalies push it outside bounds
    df['consensus_severity_label'] = df['consensus_severity_label'].clip(1, 4)

    print("\n--- Final Consensus Pseudo-Label (y*) Distribution ---")
    print(df['consensus_severity_label'].value_counts().sort_index().to_string())

    # 5. Identify the Core Target for Stage 2: Mismatch Auditing
    # A priority mismatch occurs when our objective consensus severity (1-4)
    # does not match the original human-assigned priority number.
    # Calculate severity difference
    df['severity_delta'] = (
        df['consensus_severity_label']
        - df['assigned_priority_num']
    ).abs()

    # Binary mismatch label
    df['is_mismatch'] = (
        df['severity_delta'] >= 2
    ).astype(int)

    print("\nMismatch distribution:")
    print(df['is_mismatch'].value_counts(normalize=True))

    print("\n--- Support Integrity Auditor Detection Metrics ---")
    print(f"Total Audited Tickets: {len(df)}")
    print(df['severity_delta'].value_counts().sort_index())

    print(
        df['is_mismatch']
        .value_counts(normalize=True)
        .mul(100)
    )
    print("-" * 50 + "\n")

    return df

# --- Execution ---
# Using the exact name of the file you uploaded
synthesized_df = synthesize_consensus_labels("cleaned_df_with_llm_severity.csv")

--- Starting Stage 1 Label Synthesis for: cleaned_df_with_llm_severity.csv ---
0
0

--- Final Consensus Pseudo-Label (y*) Distribution ---
consensus_severity_label
1     1515
2     6917
3    10000
4     1568

Mismatch distribution:
is_mismatch
0    0.78015
1    0.21985
Name: proportion, dtype: float64

--- Support Integrity Auditor Detection Metrics ---
Total Audited Tickets: 20000
severity_delta
0    5975
1    9628
2    4076
3     321
Name: count, dtype: int64
is_mismatch
0    78.015
1    21.985
Name: proportion, dtype: float64
--------------------------------------------------



In [17]:
synthesized_df.to_csv("synthesized_df_with_mismatch_label.csv", index=False)
print(f"\n[SUCCESS] Synthesized dataframe saved securely to: synthesized_df_with_mismatch_label.csv")


[SUCCESS] Synthesized dataframe saved securely to: synthesized_df_with_mismatch_label.csv


**STAGE 2: Training Pipeline**

In [18]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from datasets import Dataset

# ==========================================
# 1. SETUP LOGGING & CONFIGURATIONS (CODE 1)
# ==========================================
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "./sia_distilBERT_mismatch_model"
MAX_LENGTH = 320  # Slightly increased sequence max length to handle more metadata parameters safely
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

print(f"CUDA Available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"


# ==========================================
# 2. EVIDENCE DOSSIER GENERATOR (CODE 2)
# ==========================================
class EvidenceDossierGenerator:
    def __init__(self):
        # Maps integer severity states to clear operational definitions
        self.severity_map = {1: "Low", 2: "Medium", 3: "High", 4: "Critical"}

    def generate_single_dossier(self, ticket_row, prediction_prob):
        ticket_id = str(ticket_row.get('Ticket_ID', 'UNKNOWN'))
        assigned_priority = str(ticket_row.get('Priority_Level', 'Medium'))

        # Pull your true multi-signal metrics from Stage 1 calculations
        consensus_label_num = int(ticket_row.get('consensus_severity_label', 2))
        assigned_priority_num = int(ticket_row.get('assigned_priority_num', 2))

        llm_sev = int(ticket_row.get('llm_inferred_severity', 1))
        emb_sev = int(ticket_row.get('emb_inferred_severity', 1))
        nlp_sev = int(ticket_row.get('nlp_inferred_severity', 1))
        res_sev = int(ticket_row.get('res_time_inferred_severity', 1))
        emb_conf = float(ticket_row.get('emb_anchor_confidence', 0.0))

        inferred_severity = self.severity_map.get(consensus_label_num, "Medium")

        # Calculate strict directional operational variance
        delta = consensus_label_num - assigned_priority_num
        formatted_delta = f"+{delta}" if delta > 0 else str(delta)

        if delta > 0:
            mismatch_type = "Hidden Crisis"
        elif delta < 0:
            mismatch_type = "False Alarm"
        else:
            return None  # Defensive skip to ensure zero delta="0" entries slip into JSON

        # Intelligent Keyword extraction using compound category groupings
        subject_desc_text = f"{str(ticket_row.get('Ticket_Subject', ''))} {str(ticket_row.get('Ticket_Description', ''))}".lower()
        vocabulary_lexicon = {
            "critical_outage": ["outage", "crash", "down", "failure", "broken", "security", "leak", "breach"],
            "sla_escalation": ["sla", "urgent", "immediate", "overdue", "delay", "freeze", "error", "blocker"],
            "account_billing": ["billing", "charge", "invoice", "refund", "locked", "access", "login"]
        }

        detected_categories = []
        for category, terms in vocabulary_lexicon.items():
            matched = [t for t in terms if t in subject_desc_text]
            if matched:
                detected_categories.append(f"{category}:({', '.join(matched)})")

        keyword_value = " | ".join(detected_categories) if detected_categories else "Generic system transaction logs"

        # Expanded array leveraging all four distinct data streams from your file
        feature_evidence = [
            {
                "signal": "keyword",
                "value": keyword_value,
                "weight": f"{nlp_sev / 4.0:.2f}",
                "context": f"Rule-based linguistic density score mapped to severity index {nlp_sev}/4."
            },
            {
                "signal": "resolution_time",
                "value": f"{float(ticket_row.get('Resolution_Time_Hours', 0.0)):.1f} hours",
                "interpretation": f"Operational timeline outlier mapped to severity layer {res_sev}/4 proxy."
            },
            {
                "signal": "large_language_model",
                "value": f"Inferred Severity Level: {self.severity_map.get(llm_sev, 'Low')}",
                "weight": "0.65",  # Based on your exact 65% code configuration vote weight
                "analysis": "Zero-shot generative instruction fine-tuning targeted structural risk signs inside textual payload."
            },
            {
                "signal": "semantic_clustering",
                "value": f"Urgency Cluster Level: {self.severity_map.get(emb_sev, 'Low')}",
                "confidence_score": f"{emb_conf:.4f}",
                "vector_state": f"Assigned to dense proximity pocket via sentence embeddings with {emb_conf*100:.1f}% anchor convergence."
            }
        ]

        # Dynamic, context-aware reasoning engine based on actual feature interactions
        if mismatch_type == "Hidden Crisis":
            reasoning = (
                f"Operational Alert: Human agent under-classified this ticket as '{assigned_priority}'. "
                f"Our multi-signal consensus engine identified a critical bottleneck. "
                f"While the linguistic rule score was indexed at {nlp_sev}/4, the Large Language Model triggered a high-severity "
                f"flag of '{self.severity_map.get(llm_sev)}' due to structural threat indicators. "
                f"This text analysis is verified by a prolonged resolution signature ({ticket_row.get('Resolution_Time_Hours', 0.0):.1f} hours), "
                f"confirming that this ticket required high-priority resource allocation[cite: 1]."
            )
        else: # False Alarm
            reasoning = (
                f"Efficiency Alert: Human agent over-classified this ticket as '{assigned_priority}'[cite: 1]. "
                f"Ensemble auditing suggests this case represents a low-impact operational task[cite: 1]. "
                f"The semantic clustering engine associated this ticket with safe proximity anchors (Confidence: {emb_conf:.4f}), "
                f"and both the rule-based NLP layer ({nlp_sev}/4) and LLM classification downvoted the case urgency, "
                f"indicating it can be safely deprioritized[cite: 1]."
            )

        # Appending context-specific data points
        constraint_analysis = (
            f"Ticket {ticket_id} (Category: {ticket_row.get('Issue_Category', 'General')}) shows an analytical variance "
            f"of {formatted_delta} scales between human classification and our ensemble target[cite: 1]. {reasoning}"
        )

        return {
            "ticket_id": ticket_id,
            "assigned_priority": assigned_priority,
            "inferred_severity": inferred_severity,
            "mismatch_type": mismatch_type,
            "severity_delta": formatted_delta,
            "feature_evidence": feature_evidence,
            "constraint_analysis": constraint_analysis,
            "confidence": f"{prediction_prob:.4f}"
        }

    def batch_generate_dossiers(self, df, predictions, prediction_probs):
        flagged_dossiers = []
        for idx, (pred, prob) in enumerate(zip(predictions, prediction_probs)):
            if pred == 1:  # Classifier flags mismatch
                row_dict = df.iloc[idx].to_dict()
                dossier = self.generate_single_dossier(row_dict, prob)
                if dossier is not None:
                    flagged_dossiers.append(dossier)
        return flagged_dossiers

# ==========================================
# 3. DATA PREPROCESSING & RICH TEXTUAL SERIALIZATION (CODE 1)
# ==========================================
def load_and_serialize_data(csv_path):
    print("Loading and preparing dataset with expanded metadata parameters...")
    df = pd.read_csv(csv_path)

    # Feature engineering on temporal fields: extract day names to capture weekend fatigue/SLA gaps
    df['Submission_Date'] = pd.to_datetime(df['Submission_Date'])
    df['submission_day_name'] = df['Submission_Date'].dt.day_name()

    # Enhanced Serialization including Agent, Channel, and Submission Day
    def serialize_row(row):
        metadata = (
            f"[METADATA] Channel: {row['Ticket_Channel']} | "
            f"Category: {row['Issue_Category']} | "
            f"Assigned Priority: {int(row['assigned_priority_num'])} | "
            f"Tier: {int(row['customer_tier'])} | "
            f"Agent: {row['Assigned_Agent']} | "
            f"Sub-Day: {row['submission_day_name']} | "
            f"Customer Satisfaction: {row['Satisfaction_Score']} | "
            f"Resolution Time: {row['Resolution_Time_Hours']:.1f} hours"
        )
        text_data = f"[TEXT] Subject: {row['Ticket_Subject']} | Description: {row['Ticket_Description']}"
        return f"{metadata} {text_data}"

    df['serialized_input'] = df.apply(serialize_row, axis=1)

    # Isolate targets and features
    X = df['serialized_input'].tolist()
    y = df['is_mismatch'].tolist()

    # Class Weight balancing logic
    class_counts = df['is_mismatch'].value_counts()
    total_samples = len(df)
    weight_class_0 = total_samples / (2.0 * class_counts[0])
    weight_class_1 = total_samples / (2.0 * class_counts[1])
    class_weights = torch.tensor([weight_class_0, weight_class_1], dtype=torch.float).to(device)

    print(f"Computed Class Weights -> Consistent (0): {weight_class_0:.3f}, Mismatched (1): {weight_class_1:.3f}")

    return df, X, y, class_weights


# ==========================================
# 4. COMPUTE METRICS EVALUATION FUNCTION (CODE 1)
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average='macro')

    recall_consistent = recall_score(labels, predictions, pos_label=0, zero_division=0)
    recall_mismatched = recall_score(labels, predictions, pos_label=1, zero_division=0)

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "recall_consistent_class0": recall_consistent,
        "recall_mismatched_class1": recall_mismatched
    }


# ==========================================
# 5. CUSTOM TRAINER FOR PENALTY WEIGHT DISTRIBUTION (CODE 1)
# ==========================================
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


# ==========================================
# 6. EXECUTION PIPELINE (INTEGRATION HYBRID)
# ==========================================
def main():
    # Load dataset structures using Code 1 logic
    df, X, y, class_weights = load_and_serialize_data("synthesized_df_with_mismatch_label.csv")

    # Capture alignment indices to guarantee perfect dataframe matching during dossier extraction
    indices = np.arange(len(X))

    # Stratified Train-Validation Split matching Code 1 exact parameters
    X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
        X, y, indices, test_size=0.20, random_state=42, stratify=y
    )

    print(f"Train size: {len(X_train)} | Validation size: {len(X_val)}")

    # Initialize Tokenizer and build datasets using Code 1 logic
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=MAX_LENGTH)
    val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=MAX_LENGTH)

    train_dataset = Dataset.from_dict({**train_encodings, "labels": y_train})
    val_dataset = Dataset.from_dict({**val_encodings, "labels": y_val})

    # Initialize Architecture
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    model.to(device)

    # Exact training configurations from Code 1
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        bf16=torch.cuda.is_available(),
        fp16=False,
        report_to="none"
    )

    trainer = WeightedLossTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        class_weights=class_weights
    )

    print("\nBeginning DistilBERT training with advanced multi-context metadata features...")
    trainer.train()

    print("\nEvaluating final performance against validation splits...")
    eval_results = trainer.evaluate()

    print("\n================== VALIDATION METRICS RESULT ==================")
    print(f"Binary Accuracy:                 {eval_results['eval_accuracy']*100:.2f}% (Target >= 83%)")
    print(f"Macro F1-Score:                  {eval_results['eval_macro_f1']:.4f} (Target >= 0.82)")
    print(f"Class 0 Recall (Consistent):     {eval_results['eval_recall_consistent_class0']:.4f} (Target >= 0.78)")
    print(f"Class 1 Recall (Mismatched):     {eval_results['eval_recall_mismatched_class1']:.4f} (Target >= 0.78)")
    print("===============================================================")

    # Model save checkpoints from Code 1
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Expanded contextual model artifact successfully saved to '{OUTPUT_DIR}'")

    # ==========================================
    # POST-TRAINING DOSSIER GENERATION (CODE 2)
    # ==========================================
    print("\nExtracting prediction distributions for Anomaly Dossier building...")
    predictions_output = trainer.predict(val_dataset)
    logits = predictions_output.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    final_preds = np.argmax(logits, axis=-1)
    mismatch_confidences = probabilities[:, 1]  # Confidences matching Class 1 (mismatch anomalies)

    # Slice the original pandas rows matching the exact validation split position index
    val_df = df.iloc[idx_val].copy()

    dossier_engine = EvidenceDossierGenerator()
    generated_dossiers = dossier_engine.batch_generate_dossiers(
        df=val_df,
        predictions=final_preds,
        prediction_probs=mismatch_confidences
    )

    # Save output architecture directly to your target directory paths
    output_dir = "./sia_distilBERT_mismatch_model"
    os.makedirs(output_dir, exist_ok=True)
    output_file_path = os.path.join(output_dir, "evidence_dossiers.json")

    with open(output_file_path, "w", encoding="utf-8") as f:
        json.dump(generated_dossiers, f, indent=4)

    print("\n" + "="*60)
    print(" STAGE 3 DOSSIER EXPORT SUCCESSFUL")
    print("="*60)
    print(f"Total Val Anomaly Flag Entries Exported: {len(generated_dossiers)}")
    print(f"Structured Evidence File Saved to -> {output_file_path}")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

CUDA Available: True
Loading and preparing dataset with expanded metadata parameters...
Computed Class Weights -> Consistent (0): 0.641, Mismatched (1): 2.274
Train size: 16000 | Validation size: 4000


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Recall Consistent Class0,Recall Mismatched Class1
0.141954,0.188182,3,0.949250,0.927779,0.957706,0.919226



================== VALIDATION METRICS RESULT ==================
Binary Accuracy:                 94.92% (Target >= 83%)
Macro F1-Score:                  0.9278 (Target >= 0.82)
Class 0 Recall (Consistent):     0.9577 (Target >= 0.78)
Class 1 Recall (Mismatched):     0.9192 (Target >= 0.78)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Expanded contextual model artifact successfully saved to './sia_distilBERT_mismatch_model'

Extracting prediction distributions for Anomaly Dossier building...



 STAGE 3 DOSSIER EXPORT SUCCESSFUL
Total Val Anomaly Flag Entries Exported: 934
Structured Evidence File Saved to -> ./sia_distilBERT_mismatch_model/evidence_dossiers.json



In [19]:
import json

# Adjust path if your OUTPUT_DIR name is different
json_path = "./sia_distilBERT_mismatch_model/evidence_dossiers.json"

with open(json_path, "r") as f:
    dossiers = json.load(f)

print(f"Total anomalies flagged: {len(dossiers)}\n")

# Pretty-print the first dossier to audit the schema brackets
print(json.dumps(dossiers[5], indent=4))

Total anomalies flagged: 934

{
    "ticket_id": "TKT-110974",
    "assigned_priority": "Low",
    "inferred_severity": "High",
    "mismatch_type": "Hidden Crisis",
    "severity_delta": "+2",
    "feature_evidence": [
        {
            "signal": "keyword",
            "value": "sla_escalation:(error)",
            "weight": "0.50",
            "context": "Rule-based linguistic density score mapped to severity index 2/4."
        },
        {
            "signal": "resolution_time",
            "value": "120.0 hours",
            "interpretation": "Operational timeline outlier mapped to severity layer 3/4 proxy."
        },
        {
            "signal": "large_language_model",
            "value": "Inferred Severity Level: High",
            "weight": "0.65",
            "analysis": "Zero-shot generative instruction fine-tuning targeted structural risk signs inside textual payload."
        },
        {
            "signal": "semantic_clustering",
            "value": "Urgency C